# Three-Legged OAuth (3LO) with AgentCore Gateway and a Stateful MCP Server on AgentCore Runtime

This notebook demonstrates an end-to-end integration pattern where:

1. A **stateful MCP server** is deployed on [Amazon Bedrock AgentCore Runtime](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/agents-tools-runtime.html) as a Docker container, protected by a **Cognito OAuth authorizer**
2. [AgentCore Gateway](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway-target-MCPservers.html) fronts the server with **three-legged OAuth (3LO)** — the end user must authenticate via Cognito before tools can be invoked
3. The Gateway uses [header propagation](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway-headers.html) to forward `Mcp-Session-Id` for stateful session stickiness

## Three-Legged OAuth Flow

In 3LO, three parties are involved: the **client**, the **Gateway** (acting on behalf of the client), and the **authorization server** (Cognito). The flow works as follows:

1. The client authenticates to the Gateway using a Cognito M2M token (inbound auth)
2. The client calls `tools/list` — the Gateway returns cached tool definitions immediately (no 3LO needed)
3. The client calls `tools/call` — the Gateway needs an OAuth token to call the Runtime on behalf of the user
4. Since no token exists yet, the Gateway returns a **URL elicitation** with an authorization URL
5. The user opens the URL in a browser, authenticates with Cognito, and grants consent
6. The browser redirects to the AgentCore Identity callback with an authorization code
7. The client calls `CompleteResourceTokenAuth` to complete session binding
8. AgentCore Identity exchanges the code for an access token and caches it
9. The client retries `tools/call` — the Gateway uses the cached token to invoke the Runtime

```
Client              Gateway                 AgentCore Identity       Cognito          Runtime
  |                    |                          |                    |                 |
  |-- tools/call ----->|                          |                    |                 |
  |                    |-- get token ------------>|                    |                 |
  |                    |<- authorizationUrl ------|                    |                 |
  |<- URL elicitation -|                          |                    |                 |
  |                    |                          |                    |                 |
  |-- open browser ----|--------------------------|-- login + consent->|                 |
  |                    |                          |<-- auth code ------|                 |
  |                    |                          |                    |                 |
  |-- complete auth -->|-- complete binding ----->|                    |                 |
  |                    |                          |-- exchange code -->|                 |
  |                    |                          |<-- access token ---|                 |
  |                    |                          |                    |                 |
  |-- tools/call ----->|-- invoke (Bearer tok) ---|-------------------------------------->|
  |<-- result ---------|<-------------------------|<-------------------------------------|  
```

## Security Considerations

- **Inbound Auth**: Cognito M2M (client_credentials) for Gateway access. For production, use your enterprise IdP.
- **Outbound Auth**: OAuth Authorization Code grant via [AgentCore Identity](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/resource-providers.html) credential provider.
- **Session Binding**: [URL session binding](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/oauth2-authorization-url-session-binding.html) ensures the user who initiated the flow is the same user who granted consent.
- **Least Privilege IAM**: All roles are scoped to specific resources. The Gateway role requires `GetWorkloadAccessTokenForJWT` for the 3LO token exchange.
- **Runtime Protection**: The Runtime uses a `CustomJWTAuthorizer` with Cognito, accepting only tokens from the configured client.

## Prerequisites

- Python 3.10+ with `boto3 >= 1.38.0` and `requests`
- Docker installed and running
- AWS credentials configured (`aws configure`)


Copyright Amazon.com, Inc. or its affiliates. All Rights Reserved. SPDX-License-Identifier: Apache-2.0

## Setup

In [ ]:
import sys
!{sys.executable} -m pip install 'boto3>=1.38.0' 'botocore>=1.38.0' requests -q


In [ ]:
import boto3
import json
import os
import subprocess
import base64
import requests
import time
import uuid

REGION = boto3.session.Session().region_name or 'us-east-1'
os.environ['AWS_DEFAULT_REGION'] = REGION
ACCOUNT_ID = boto3.client('sts').get_caller_identity()['Account']

agentcore = boto3.client('bedrock-agentcore-control')
cognito = boto3.client('cognito-idp', region_name=REGION)
iam = boto3.client('iam')
ecr = boto3.client('ecr')

print(f'Region:  {REGION}')
print(f'Account: {ACCOUNT_ID}')


---
## Step 1: Write the Stateful MCP Server

We create a Task Tracker MCP server with `stateless_http=False` to enable stateful features:
session-scoped state, elicitation, and progress notifications.

The server, its requirements, and a Dockerfile are written to a `build/` directory for container packaging.

In [ ]:
BUILD_DIR = 'build'
os.makedirs(BUILD_DIR, exist_ok=True)

# ── MCP Server Code ──
SERVER_CODE = '''\
"""Task Tracker - Stateful MCP Server."""
import asyncio, json, uuid
from datetime import datetime, timezone
from enum import Enum
from typing import Optional
from fastmcp import FastMCP, Context

mcp = FastMCP("Task-Tracker")
_session_tasks: dict[str, list[dict]] = {}

def _get_tasks(sid: str) -> list[dict]:
    if sid not in _session_tasks:
        _session_tasks[sid] = []
    return _session_tasks[sid]

class Priority(str, Enum):
    LOW = "low"
    MEDIUM = "medium"
    HIGH = "high"
    CRITICAL = "critical"

@mcp.resource("tasks://list")
def resource_task_list() -> str:
    return json.dumps({s[:8]: t for s, t in _session_tasks.items()}, indent=2, default=str)

@mcp.tool()
async def add_task(ctx: Context, title: str, priority: str = "medium",
                   assignee: Optional[str] = None, description: Optional[str] = None) -> str:
    """Add a new task to the session task list."""
    tasks = _get_tasks(ctx.session_id)
    task = {"id": str(uuid.uuid4())[:8], "title": title, "priority": priority,
            "status": "todo", "assignee": assignee, "description": description,
            "created_at": datetime.now(timezone.utc).isoformat()}
    tasks.append(task)
    return json.dumps({"message": f"Task created: {title}", "task": task}, indent=2)

@mcp.tool()
async def list_tasks(ctx: Context, status_filter: Optional[str] = None) -> str:
    """List all tasks, optionally filtered by status (todo, in_progress, done)."""
    tasks = _get_tasks(ctx.session_id)
    filtered = [t for t in tasks if not status_filter or t["status"] == status_filter]
    return json.dumps({"count": len(filtered), "tasks": filtered}, indent=2, default=str)

@mcp.tool()
async def update_task_status(ctx: Context, task_id: str, new_status: str) -> str:
    """Update a task status to: todo, in_progress, or done."""
    for task in _get_tasks(ctx.session_id):
        if task["id"] == task_id:
            old = task["status"]
            task["status"] = new_status
            task["updated_at"] = datetime.now(timezone.utc).isoformat()
            return json.dumps({"message": f"{old} -> {new_status}", "task": task}, indent=2, default=str)
    return json.dumps({"error": f"Task {task_id} not found"})

@mcp.tool()
async def create_task_interactive(ctx: Context) -> str:
    """Create a task interactively via elicitation."""
    r = await ctx.elicit(message="Task title?", response_type=str)
    if r.action != "accept": return "Cancelled."
    title = r.data
    r = await ctx.elicit(message="Priority? (low/medium/high/critical)", response_type=Priority)
    if r.action != "accept": return "Cancelled."
    priority = r.data
    r = await ctx.elicit(message="Assignee? (or skip)", response_type=str)
    assignee = r.data if r.action == "accept" and r.data.lower() != "skip" else None
    tasks = _get_tasks(ctx.session_id)
    task = {"id": str(uuid.uuid4())[:8], "title": title, "priority": priority,
            "status": "todo", "assignee": assignee,
            "created_at": datetime.now(timezone.utc).isoformat()}
    tasks.append(task)
    return json.dumps({"message": f"Task created: {title}", "task": task}, indent=2, default=str)

@mcp.tool()
async def generate_report(ctx: Context) -> str:
    """Generate a task summary report with progress notifications."""
    tasks = _get_tasks(ctx.session_id)
    for step in range(1, 5):
        await ctx.report_progress(progress=step, total=4)
        await asyncio.sleep(0.2)
    by_status, by_priority = {}, {}
    for t in tasks:
        by_status[t["status"]] = by_status.get(t["status"], 0) + 1
        by_priority[t["priority"]] = by_priority.get(t["priority"], 0) + 1
    return json.dumps({"report": {
        "total": len(tasks), "by_status": by_status, "by_priority": by_priority,
    }}, indent=2)

if __name__ == "__main__":
    mcp.run(transport="streamable-http", host="0.0.0.0", port=8000, stateless_http=False)
'''

with open(f'{BUILD_DIR}/task_tracker_server.py', 'w') as f:
    f.write(SERVER_CODE)
print(f'Written: {BUILD_DIR}/task_tracker_server.py')

with open(f'{BUILD_DIR}/requirements.txt', 'w') as f:
    f.write('fastmcp>=2.10.0\nmcp\n')
print(f'Written: {BUILD_DIR}/requirements.txt')

with open(f'{BUILD_DIR}/Dockerfile', 'w') as f:
    f.write('FROM --platform=linux/arm64 python:3.13-slim\n'
            'WORKDIR /app\n'
            'COPY requirements.txt .\n'
            'RUN pip install --no-cache-dir -r requirements.txt\n'
            'COPY task_tracker_server.py .\n'
            'EXPOSE 8000\n'
            'CMD ["python", "task_tracker_server.py"]\n')
print(f'Written: {BUILD_DIR}/Dockerfile')


---
## Step 2: Build Docker Image and Push to ECR

AgentCore Runtime requires ARM64 container images. Pre-installing dependencies in the image
ensures fast cold starts.

In [ ]:
ECR_REPO = 'agentcore/tasktracker3lo'
IMAGE_TAG = 'latest'

try:
    ecr.create_repository(repositoryName=ECR_REPO)
    print(f'Created ECR repo: {ECR_REPO}')
except ecr.exceptions.RepositoryAlreadyExistsException:
    print(f'ECR repo exists: {ECR_REPO}')

container_uri = f'{ACCOUNT_ID}.dkr.ecr.{REGION}.amazonaws.com/{ECR_REPO}:{IMAGE_TAG}'

# Docker login to ECR
token = ecr.get_authorization_token()['authorizationData'][0]
user, passwd = base64.b64decode(token['authorizationToken']).decode().split(':')
subprocess.run(['docker', 'login', '--username', user, '--password-stdin', token['proxyEndpoint']],
               input=passwd, text=True, capture_output=True, check=True)
print('Docker logged in to ECR')

print('Building ARM64 image (this may take a minute)...')
subprocess.run(['docker', 'build', '--platform', 'linux/arm64', '-t', container_uri, '.'],
               cwd=BUILD_DIR, check=True, capture_output=True)
print(f'Built: {container_uri}')

print('Pushing to ECR...')
subprocess.run(['docker', 'push', container_uri], check=True, capture_output=True)
print(f'Pushed: {container_uri}')


---
## Step 3: Create IAM Roles

Two roles are needed:
- **Runtime execution role** — ECR pull + CloudWatch logs
- **Gateway service role** — Runtime invocation + workload identity management. Note: `GetWorkloadAccessTokenForJWT` is required for the 3LO token exchange.

Both use `bedrock-agentcore.amazonaws.com` as the trust principal.

In [ ]:
TRUST_POLICY = json.dumps({'Version': '2012-10-17', 'Statement': [
    {'Effect': 'Allow', 'Principal': {'Service': 'bedrock-agentcore.amazonaws.com'}, 'Action': 'sts:AssumeRole'}]})

def create_role(name, policy, desc):
    try:
        iam.create_role(RoleName=name, AssumeRolePolicyDocument=TRUST_POLICY, Description=desc)
        print(f'Created: {name}')
    except iam.exceptions.EntityAlreadyExistsException:
        print(f'Exists: {name}')
    iam.put_role_policy(RoleName=name, PolicyName=f'{name}-policy', PolicyDocument=json.dumps(policy))
    return iam.get_role(RoleName=name)['Role']['Arn']

RUNTIME_ROLE = 'tasktracker3lo-rt-role'
runtime_role_arn = create_role(RUNTIME_ROLE,
    {'Version': '2012-10-17', 'Statement': [
        {'Effect': 'Allow', 'Action': ['ecr:GetAuthorizationToken', 'ecr:BatchGetImage',
            'ecr:GetDownloadUrlForLayer'], 'Resource': '*'},
        {'Effect': 'Allow', 'Action': ['logs:CreateLogGroup', 'logs:CreateLogStream',
            'logs:PutLogEvents'], 'Resource': '*'},
    ]}, 'Runtime execution role')

GATEWAY_ROLE = 'tasktracker3lo-gw-role'
gateway_role_arn = create_role(GATEWAY_ROLE,
    {'Version': '2012-10-17', 'Statement': [
        {'Effect': 'Allow', 'Action': ['bedrock-agentcore:InvokeAgentRuntime',
            'bedrock-agentcore:InvokeAgent', 'bedrock-agentcore:GetAgentRuntimeEndpoint',
            'bedrock-agentcore:GetAgentRuntime'], 'Resource': '*'},
        {'Effect': 'Allow', 'Action': ['bedrock-agentcore:CreateWorkloadIdentity',
            'bedrock-agentcore:GetWorkloadAccessToken',
            'bedrock-agentcore:GetWorkloadAccessTokenForUserId',
            'bedrock-agentcore:GetWorkloadAccessTokenForJWT',
            'bedrock-agentcore:GetResourceOauth2Token',
            'bedrock-agentcore:CompleteResourceTokenAuth',
            'secretsmanager:GetSecretValue'], 'Resource': '*'},
    ]}, 'Gateway service role (3LO)')

print(f'Runtime role: {runtime_role_arn}')
print(f'Gateway role: {gateway_role_arn}')
print('Waiting 15s for IAM propagation...')
time.sleep(15)


---
## Step 4: Setup Cognito for Runtime OAuth (Outbound)

This Cognito user pool protects the Runtime MCP server. The Gateway will obtain tokens from this pool
via the Authorization Code grant to invoke the Runtime on behalf of the end user.

We create:
- A user pool with a domain (required for the hosted UI login page)
- An app client with `code` flow (Authorization Code grant)
- A test user for the 3LO consent flow

In [ ]:
RT_POOL_NAME = 'tasktracker3lo-rt-pool'

# Get or create the user pool
rt_pool_id = None
for p in cognito.get_paginator('list_user_pools').paginate(MaxResults=60):
    for pool in p['UserPools']:
        if pool['Name'] == RT_POOL_NAME:
            rt_pool_id = pool['Id']
if not rt_pool_id:
    rt_pool_id = cognito.create_user_pool(PoolName=RT_POOL_NAME)['UserPool']['Id']
    domain = 'tasktracker3lo-' + rt_pool_id.split('_')[1][:8].lower()
    try:
        cognito.create_user_pool_domain(Domain=domain, UserPoolId=rt_pool_id)
    except Exception:
        pass

rt_pool_desc = cognito.describe_user_pool(UserPoolId=rt_pool_id)['UserPool']
rt_domain = rt_pool_desc.get('Domain')
rt_discovery = f'https://cognito-idp.{REGION}.amazonaws.com/{rt_pool_id}/.well-known/openid-configuration'

# Get or create the Authorization Code client
RT_CLIENT_NAME = 'tasktracker3lo-rt-client'
rt_cid = rt_csec = None
for p in cognito.get_paginator('list_user_pool_clients').paginate(UserPoolId=rt_pool_id, MaxResults=60):
    for c in p['UserPoolClients']:
        if c['ClientName'] == RT_CLIENT_NAME:
            d = cognito.describe_user_pool_client(UserPoolId=rt_pool_id, ClientId=c['ClientId'])['UserPoolClient']
            rt_cid, rt_csec = d['ClientId'], d.get('ClientSecret', '')
if not rt_cid:
    r = cognito.create_user_pool_client(
        UserPoolId=rt_pool_id, ClientName=RT_CLIENT_NAME, GenerateSecret=True,
        AllowedOAuthFlows=['code'], AllowedOAuthScopes=['openid'],
        AllowedOAuthFlowsUserPoolClient=True,
        CallbackURLs=['http://localhost:8080/callback'],
        SupportedIdentityProviders=['COGNITO'])
    rt_cid, rt_csec = r['UserPoolClient']['ClientId'], r['UserPoolClient']['ClientSecret']

# Create a test user
try:
    cognito.admin_create_user(UserPoolId=rt_pool_id, Username='testuser',
        TemporaryPassword='Temp1234!', MessageAction='SUPPRESS')
    cognito.admin_set_user_password(UserPoolId=rt_pool_id, Username='testuser',
        Password='Test1234!', Permanent=True)
    print('Created test user: testuser / Test1234!')
except cognito.exceptions.UsernameExistsException:
    print('Test user exists: testuser / Test1234!')

print(f'Runtime Cognito pool: {rt_pool_id}')
print(f'Runtime client: {rt_cid}')
print(f'Runtime domain: {rt_domain}')
print(f'Discovery URL: {rt_discovery}')


---
## Step 5: Deploy to AgentCore Runtime with Cognito Authorizer

The Runtime is deployed with a `customJWTAuthorizer` pointing to the Cognito pool created above.
This means the Runtime will only accept requests with a valid JWT from that pool — the Gateway
must obtain such a token via the 3LO flow before it can invoke tools.

In [ ]:
RUNTIME_NAME = 'tasktracker3lo'

# Check if runtime already exists
runtime_id = runtime_arn = None
for rt in agentcore.list_agent_runtimes(maxResults=100).get('agentRuntimes', []):
    if rt['agentRuntimeName'] == RUNTIME_NAME:
        runtime_id, runtime_arn = rt['agentRuntimeId'], rt['agentRuntimeArn']
        print(f'Runtime exists: {runtime_id} ({rt["status"]})')

if not runtime_id:
    r = agentcore.create_agent_runtime(
        agentRuntimeName=RUNTIME_NAME,
        roleArn=runtime_role_arn,
        agentRuntimeArtifact={'containerConfiguration': {'containerUri': container_uri}},
        networkConfiguration={'networkMode': 'PUBLIC'},
        protocolConfiguration={'serverProtocol': 'MCP'},
        authorizerConfiguration={
            'customJWTAuthorizer': {
                'allowedClients': [rt_cid],
                'discoveryUrl': rt_discovery,
            }
        },
    )
    runtime_id, runtime_arn = r['agentRuntimeId'], r['agentRuntimeArn']
    print(f'Created runtime: {runtime_id}')

# Wait for READY
for i in range(30):
    s = agentcore.get_agent_runtime(agentRuntimeId=runtime_id)['status']
    if s in ('ACTIVE', 'READY'):
        print(f'Runtime: {s}')
        break
    print(f'  [{i+1}] {s}')
    time.sleep(10)

# Create DEFAULT endpoint
try:
    agentcore.create_agent_runtime_endpoint(agentRuntimeId=runtime_id, name='DEFAULT')
    print('Created endpoint: DEFAULT')
except Exception as e:
    if 'Conflict' in str(e) or 'already' in str(e).lower():
        print('Endpoint DEFAULT exists')
    else:
        raise

for i in range(15):
    s = agentcore.get_agent_runtime_endpoint(agentRuntimeId=runtime_id, endpointName='DEFAULT')['status']
    if s in ('ACTIVE', 'READY'):
        break
    time.sleep(5)

encoded_arn = runtime_arn.replace(':', '%3A').replace('/', '%2F')
RUNTIME_ENDPOINT = f'https://bedrock-agentcore.{REGION}.amazonaws.com/runtimes/{encoded_arn}/invocations?qualifier=DEFAULT'
print(f'\nRuntime ARN: {runtime_arn}')
print(f'Endpoint: {RUNTIME_ENDPOINT}')


---
## Step 6: Create AgentCore Identity Credential Provider

The credential provider connects the Gateway to the Runtime's Cognito pool. It uses `CustomOauth2`
vendor with the Cognito discovery URL. When the Gateway needs a token for the Runtime, it goes
through this credential provider to initiate the Authorization Code flow.

**Important:** After creating the credential provider, update the Cognito client's callback URL
to include the AgentCore Identity callback URL returned by the API.

In [ ]:
CRED_PROVIDER_NAME = 'tasktracker3lo-cred'

# Delete if exists (for idempotency)
try:
    agentcore.delete_oauth2_credential_provider(name=CRED_PROVIDER_NAME)
    time.sleep(2)
except Exception:
    pass

cred_resp = agentcore.create_oauth2_credential_provider(
    name=CRED_PROVIDER_NAME,
    credentialProviderVendor='CustomOauth2',
    oauth2ProviderConfigInput={
        'customOauth2ProviderConfig': {
            'clientId': rt_cid,
            'clientSecret': rt_csec,
            'oauthDiscovery': {'discoveryUrl': rt_discovery},
        }
    },
)

cred_provider_arn = cred_resp['credentialProviderArn']
identity_callback = cred_resp['callbackUrl']

print(f'Credential provider: {cred_provider_arn}')
print(f'Callback URL: {identity_callback}')

# Update Cognito client with the AgentCore Identity callback
cognito.update_user_pool_client(
    UserPoolId=rt_pool_id, ClientId=rt_cid,
    AllowedOAuthFlows=['code'], AllowedOAuthScopes=['openid'],
    AllowedOAuthFlowsUserPoolClient=True,
    CallbackURLs=[identity_callback, 'http://localhost:8080/callback'],
    SupportedIdentityProviders=['COGNITO'])
print('Updated Cognito client callback URLs')


---
## Step 7: Create AgentCore Gateway with Cognito Inbound Auth

The Gateway uses a separate Cognito pool for inbound M2M authentication (client_credentials grant).
This controls who can call the Gateway. The outbound 3LO flow (Authorization Code) is configured
on the target in the next step.

In [ ]:
# ── Inbound Cognito (M2M) ──
GW_POOL_NAME = 'tasktracker3lo-gw-pool'
GW_RS_ID = 'tasktracker3lo-gw-rs'
GW_CLIENT_NAME = 'tasktracker3lo-gw-client'

gw_pool_id = None
for p in cognito.get_paginator('list_user_pools').paginate(MaxResults=60):
    for pool in p['UserPools']:
        if pool['Name'] == GW_POOL_NAME:
            gw_pool_id = pool['Id']
if not gw_pool_id:
    gw_pool_id = cognito.create_user_pool(PoolName=GW_POOL_NAME)['UserPool']['Id']
    gw_domain = 'tasktracker3lo-gw-' + gw_pool_id.split('_')[1][:8].lower()
    try:
        cognito.create_user_pool_domain(Domain=gw_domain, UserPoolId=gw_pool_id)
    except Exception:
        pass

try:
    cognito.describe_resource_server(UserPoolId=gw_pool_id, Identifier=GW_RS_ID)
except cognito.exceptions.ResourceNotFoundException:
    cognito.create_resource_server(UserPoolId=gw_pool_id, Identifier=GW_RS_ID, Name=GW_RS_ID,
        Scopes=[{'ScopeName': 'invoke', 'ScopeDescription': 'Invoke Gateway'}])

gw_cid = gw_csec = None
for p in cognito.get_paginator('list_user_pool_clients').paginate(UserPoolId=gw_pool_id, MaxResults=60):
    for c in p['UserPoolClients']:
        if c['ClientName'] == GW_CLIENT_NAME:
            d = cognito.describe_user_pool_client(UserPoolId=gw_pool_id, ClientId=c['ClientId'])['UserPoolClient']
            gw_cid, gw_csec = d['ClientId'], d.get('ClientSecret', '')
if not gw_cid:
    r = cognito.create_user_pool_client(
        UserPoolId=gw_pool_id, ClientName=GW_CLIENT_NAME, GenerateSecret=True,
        AllowedOAuthFlows=['client_credentials'], AllowedOAuthScopes=[f'{GW_RS_ID}/invoke'],
        AllowedOAuthFlowsUserPoolClient=True)
    gw_cid, gw_csec = r['UserPoolClient']['ClientId'], r['UserPoolClient']['ClientSecret']

gw_discovery = f'https://cognito-idp.{REGION}.amazonaws.com/{gw_pool_id}/.well-known/openid-configuration'
gw_scope = f'{GW_RS_ID}/invoke'

print(f'Inbound pool: {gw_pool_id}')
print(f'Inbound client: {gw_cid}')

# ── Create Gateway ──
GATEWAY_NAME = 'tasktracker3lo-gw'
gateway_id = gateway_url = None
for gw in agentcore.list_gateways().get('items', []):
    if gw['name'] == GATEWAY_NAME:
        d = agentcore.get_gateway(gatewayIdentifier=gw['gatewayId'])
        gateway_id, gateway_url = gw['gatewayId'], d['gatewayUrl']
        print(f'Gateway exists: {gateway_id}')

if not gateway_id:
    r = agentcore.create_gateway(
        name=GATEWAY_NAME, roleArn=gateway_role_arn, protocolType='MCP',
        protocolConfiguration={'mcp': {'supportedVersions': ['2025-11-25'], 'searchType': 'SEMANTIC'}},
        authorizerType='CUSTOM_JWT',
        authorizerConfiguration={'customJWTAuthorizer': {'allowedClients': [gw_cid], 'discoveryUrl': gw_discovery}})
    gateway_id = r['gatewayId']
    print(f'Created gateway: {gateway_id}')
    for i in range(20):
        gw = agentcore.get_gateway(gatewayIdentifier=gateway_id)
        if gw['status'] in ('ACTIVE', 'READY'):
            gateway_url = gw['gatewayUrl']
            break
        time.sleep(5)

print(f'Gateway URL: {gateway_url}')


---
## Step 8: Add Runtime Target with 3LO (Authorization Code Grant)

The target is configured with:
- **OAuth credential provider** with `AUTHORIZATION_CODE` grant — triggers the 3LO flow on `tools/call`
- **Tool schema upfront** (`mcpToolSchema`) — avoids needing admin auth during target creation
- **Header propagation** for `Mcp-Session-Id` — enables stateful session stickiness through the Gateway

In [ ]:
# Tool schema provided upfront so the Gateway caches tools without connecting to the Runtime
tool_schema = json.dumps({'tools': [
    {'name': 'add_task', 'description': 'Add a new task.', 'inputSchema': {'type': 'object',
        'properties': {'title': {'type': 'string'}, 'priority': {'type': 'string'},
            'assignee': {'type': 'string'}, 'description': {'type': 'string'}}, 'required': ['title']}},
    {'name': 'list_tasks', 'description': 'List all tasks.', 'inputSchema': {'type': 'object',
        'properties': {'status_filter': {'type': 'string'}}}},
    {'name': 'update_task_status', 'description': 'Update task status.', 'inputSchema': {'type': 'object',
        'properties': {'task_id': {'type': 'string'}, 'new_status': {'type': 'string'}}, 'required': ['task_id', 'new_status']}},
    {'name': 'create_task_interactive', 'description': 'Create task via elicitation.', 'inputSchema': {'type': 'object', 'properties': {}}},
    {'name': 'generate_report', 'description': 'Generate report with progress.', 'inputSchema': {'type': 'object', 'properties': {}}},
]})

target_name = f'tasktracker3lo-{str(uuid.uuid4())[:4]}'
r = agentcore.create_gateway_target(
    gatewayIdentifier=gateway_id, name=target_name,
    targetConfiguration={'mcp': {'mcpServer': {'endpoint': RUNTIME_ENDPOINT,
        'mcpToolSchema': {'inlinePayload': tool_schema}}}},
    credentialProviderConfigurations=[{
        'credentialProviderType': 'OAUTH',
        'credentialProvider': {'oauthCredentialProvider': {
            'providerArn': cred_provider_arn,
            'grantType': 'AUTHORIZATION_CODE',
            'defaultReturnUrl': 'http://localhost:8080/callback',
            'scopes': ['openid'],
        }},
    }],
    metadataConfiguration={
        'allowedRequestHeaders': ['Mcp-Session-Id'],
        'allowedResponseHeaders': ['Mcp-Session-Id'],
    },
)
target_id = r['targetId']
print(f'Target: {target_name} ({target_id}) - {r["status"]}')

for i in range(20):
    d = agentcore.get_gateway_target(gatewayIdentifier=gateway_id, targetId=target_id)
    s = d['status']
    if s in ('ACTIVE', 'READY'):
        print(f'Target: {s}')
        break
    if 'UNSUCCESSFUL' in s or 'FAILED' in s:
        print(f'{s}: {d.get("statusReasons", [])}')
        break
    print(f'  [{i+1}] {s}')
    time.sleep(10)


---
## Step 9: Invoke Through the Gateway

### 9.1 Authenticate (inbound)

Get a Cognito M2M token for Gateway access.

In [ ]:
gw_pool_desc = cognito.describe_user_pool(UserPoolId=gw_pool_id)['UserPool']
gw_domain_name = gw_pool_desc['Domain']

tok_resp = requests.post(
    f'https://{gw_domain_name}.auth.{REGION}.amazoncognito.com/oauth2/token',
    data={'grant_type': 'client_credentials', 'scope': gw_scope},
    auth=(gw_cid, gw_csec),
    headers={'Content-Type': 'application/x-www-form-urlencoded'})
tok_resp.raise_for_status()
token = tok_resp.json()['access_token']

headers = {'Content-Type': 'application/json', 'Accept': 'application/json, text/event-stream',
           'Authorization': f'Bearer {token}', 'Mcp-Protocol-Version': '2025-11-25'}
print('Authenticated with Gateway.')


### 9.2 List tools (cached, no 3LO needed)

The Gateway returns cached tool definitions from the schema provided during target creation.

In [ ]:
r = requests.post(gateway_url, headers=headers,
    json={'jsonrpc': '2.0', 'id': 'list', 'method': 'tools/list'})
print(json.dumps(r.json(), indent=2))


### 9.3 Call a tool and complete the 3LO flow

This cell does everything in one shot to avoid auth URL expiration (10 min limit):
1. Starts a local callback server on port 8080
2. Calls `tools/call` which triggers the 3LO elicitation
3. Opens the authorization URL in your browser
4. Waits for you to log in (testuser / Test1234!) and captures the callback
5. Completes session binding via `CompleteResourceTokenAuth`
6. Retries the tool call with the cached token

In [ ]:
import webbrowser
from http.server import HTTPServer, BaseHTTPRequestHandler
from urllib.parse import urlparse, parse_qs
import threading

# Kill any leftover process on port 8080
import signal
try:
    import subprocess as sp
    pids = sp.run(['lsof', '-ti:8080'], capture_output=True, text=True).stdout.strip()
    if pids:
        for pid in pids.split():
            os.kill(int(pid), signal.SIGKILL)
except Exception:
    pass

# 1. Start callback server FIRST
callback_data = {}
callback_received = threading.Event()

class CallbackHandler(BaseHTTPRequestHandler):
    def do_GET(self):
        parsed = urlparse(self.path)
        callback_data.update(parse_qs(parsed.query))
        self.send_response(200)
        self.send_header('Content-Type', 'text/html')
        self.end_headers()
        self.wfile.write(b'<h1>Authorization complete!</h1><p>Return to the notebook.</p>')
        callback_received.set()
    def log_message(self, *args): pass

server = HTTPServer(('localhost', 8080), CallbackHandler)
server_thread = threading.Thread(target=server.handle_request, daemon=True)
server_thread.start()
print('1. Callback server started on http://localhost:8080')

# 2. Call tools/call to trigger 3LO elicitation
print('2. Calling tools/call to trigger 3LO...')
r = requests.post(gateway_url, headers=headers, json={
    'jsonrpc': '2.0', 'id': 'call-3lo', 'method': 'tools/call',
    'params': {'name': f'{target_name}___add_task', 'arguments': {'title': 'Test 3LO', 'priority': 'high'}}
})
result = r.json()

if 'error' in result and result['error'].get('code') == -32042:
    elicitation = result['error']['data']['elicitations'][0]
    auth_url = elicitation['url']
    print(f'3. Got authorization URL, opening browser...')
    webbrowser.open(auth_url)
    
    # 3. Wait for callback
    print('4. Waiting for authentication (log in with testuser / Test1234!)...')
    callback_received.wait(timeout=300)
    server.server_close()
    
    if callback_data:
        print(f'5. Callback received: {list(callback_data.keys())}')
        
        # 4. Complete session binding
        agentcore_dp = boto3.client('bedrock-agentcore')
        session_uri = (callback_data.get('session_id') or callback_data.get('session-uri') or callback_data.get('session_uri') or [None])[0]
        if session_uri:
            try:
                agentcore_dp.complete_resource_token_auth(
                    sessionUri=session_uri, userIdentifier={'userToken': token})
                print('6. Session binding completed!')
            except Exception as e:
                print(f'Session binding note: {e}')
        else:
            print(f'5. No session URI — token may be cached. Params: {callback_data}')
        
        # 5. Retry the tool call
        print('\n--- Retrying tools/call ---')
        r = requests.post(gateway_url, headers=headers, json={
            'jsonrpc': '2.0', 'id': 'call-after-3lo', 'method': 'tools/call',
            'params': {'name': f'{target_name}___add_task', 'arguments': {'title': 'First task via 3LO', 'priority': 'high'}}
        })
        print(json.dumps(r.json(), indent=2))
    else:
        print('Timeout waiting for callback.')
        server.server_close()
else:
    server.server_close()
    print('Unexpected response (not a 3LO elicitation):')
    print(json.dumps(result, indent=2))


### 9.5 Retry the tool call

After completing the 3LO flow, the Gateway has a cached token. Subsequent `tools/call` requests
should succeed without triggering the authorization flow again.

In [ ]:
# Retry — should succeed now that the 3LO flow is complete
r = requests.post(gateway_url, headers=headers, json={
    'jsonrpc': '2.0', 'id': 'call-after-3lo', 'method': 'tools/call',
    'params': {'name': f'{target_name}___add_task', 'arguments': {'title': 'First task via 3LO', 'priority': 'high'}}
})
print(json.dumps(r.json(), indent=2))


---
## Step 10: Demonstrate Stateful Features Through the Gateway

Now that the 3LO flow is complete, we demonstrate the stateful capabilities of the MCP server.
The key to statefulness through the Gateway is the `Mcp-Session-Id` header, which is propagated
between client and Runtime via the `allowedRequestHeaders` / `allowedResponseHeaders` configuration.

We will:
1. Inspect the `Mcp-Session-Id` returned by the Runtime
2. Add multiple tasks across separate `tools/call` requests
3. List all tasks (proving session state persists across calls)
4. Update a task status
5. Generate a report with progress notifications

### 10.1 Establish session and inspect `Mcp-Session-Id`

In [ ]:
# Helper to call tools and propagate Mcp-Session-Id
def call_tool(tool, args, call_id='call'):
    body = {'jsonrpc': '2.0', 'id': call_id, 'method': 'tools/call',
            'params': {'name': f'{target_name}___{tool}', 'arguments': args}}
    r = requests.post(gateway_url, headers=headers, json=body)
    # Capture Mcp-Session-Id for session stickiness
    sid = r.headers.get('Mcp-Session-Id')
    if sid:
        headers['Mcp-Session-Id'] = sid
    return r.json()

# Make a lightweight call to establish the session
result = call_tool('list_tasks', {}, 'session-probe')

session_id = headers.get('Mcp-Session-Id')
if session_id:
    print(f'Mcp-Session-Id: {session_id}')
    print('Session established — all subsequent requests will hit the same Runtime microVM.')
else:
    print('No Mcp-Session-Id returned.')


### 10.2 Add tasks across separate calls

Each `tools/call` goes through the Gateway to the same Runtime microVM (via `Mcp-Session-Id`).
Tasks are stored in the server's in-memory session-scoped dictionary.

In [ ]:
for title, priority, assignee in [
    ('Set up CI/CD pipeline', 'high', 'alice'),
    ('Write API documentation', 'medium', 'bob'),
    ('Fix production security vulnerability', 'critical', 'alice'),
]:
    result = call_tool('add_task', {'title': title, 'priority': priority, 'assignee': assignee})
    task = json.loads(result['result']['content'][0]['text'])
    print(f"  Added: {task['task']['title']} (id={task['task']['id']})")


### 10.3 List all tasks (proves statefulness)

All three tasks from previous calls should be returned, proving that session state
persists across separate `tools/call` requests through the Gateway.

In [ ]:
result = call_tool('list_tasks', {}, 'list-all')
data = json.loads(result['result']['content'][0]['text'])
print(f"Task count: {data['count']}")
for t in data['tasks']:
    print(f"  [{t['id']}] {t['title']} - {t['status']} ({t['priority']})")

assert data['count'] >= 3, f"Expected at least 3 tasks, got {data['count']} — statefulness may not be working"
print(f'\nStatefulness confirmed: {data["count"]} tasks persisted across separate calls.')

# Save a task ID for the update step
first_task_id = data['tasks'][0]['id']


### 10.4 Update task status

In [ ]:
result = call_tool('update_task_status', {'task_id': first_task_id, 'new_status': 'in_progress'}, 'update')
update = json.loads(result['result']['content'][0]['text'])
print(f"Updated: {update['message']}")


### 10.5 Generate report (progress notifications)

The `generate_report` tool sends progress notifications during execution.
In a streaming client, these appear as real-time updates.

In [ ]:
result = call_tool('generate_report', {}, 'report')
report = json.loads(result['result']['content'][0]['text'])['report']
print(f"Total tasks: {report['total']}")
print(f"By status:   {report['by_status']}")
print(f"By priority: {report['by_priority']}")


### 10.6 Filter tasks by status

Show only in-progress tasks.

In [ ]:
result = call_tool('list_tasks', {'status_filter': 'in_progress'}, 'filter')
data = json.loads(result['result']['content'][0]['text'])
print(f"In-progress tasks: {data['count']}")
for t in data['tasks']:
    print(f"  [{t['id']}] {t['title']}")


---
## Cleanup

Remove all AWS resources created in this notebook.

In [ ]:
# 1. Delete Gateway targets first, then the Gateway
try:
    targets = agentcore.list_gateway_targets(gatewayIdentifier=gateway_id).get('items', [])
    for t in targets:
        agentcore.delete_gateway_target(gatewayIdentifier=gateway_id, targetId=t['targetId'])
        print(f'Deleted target: {t["targetId"]}')
    if targets:
        print('Waiting for targets to be deleted...')
        time.sleep(10)
    agentcore.delete_gateway(gatewayIdentifier=gateway_id)
    print(f'Deleted gateway: {gateway_id}')
except Exception as e:
    print(f'Gateway cleanup: {e}')

# 2. Delete credential provider
try:
    agentcore.delete_oauth2_credential_provider(name=CRED_PROVIDER_NAME)
    print(f'Deleted credential provider: {CRED_PROVIDER_NAME}')
except Exception as e:
    print(f'Cred provider cleanup: {e}')

# 3. Delete IAM roles
for role in [RUNTIME_ROLE, GATEWAY_ROLE]:
    try:
        for p in iam.list_role_policies(RoleName=role)['PolicyNames']:
            iam.delete_role_policy(RoleName=role, PolicyName=p)
        iam.delete_role(RoleName=role)
        print(f'Deleted role: {role}')
    except Exception as e:
        print(f'Role cleanup ({role}): {e}')

# 4. Delete Cognito pools
for pool_id_to_delete in [rt_pool_id, gw_pool_id]:
    try:
        desc = cognito.describe_user_pool(UserPoolId=pool_id_to_delete)['UserPool']
        if desc.get('Domain'):
            cognito.delete_user_pool_domain(Domain=desc['Domain'], UserPoolId=pool_id_to_delete)
        cognito.delete_user_pool(UserPoolId=pool_id_to_delete)
        print(f'Deleted pool: {pool_id_to_delete}')
    except Exception as e:
        print(f'Pool cleanup: {e}')

# 5. Delete Runtime
try:
    agentcore.delete_agent_runtime(agentRuntimeId=runtime_id)
    print(f'Deleted runtime: {runtime_id}')
except Exception as e:
    print(f'Runtime cleanup: {e}')

# 6. Delete ECR repo
try:
    ecr.delete_repository(repositoryName=ECR_REPO, force=True)
    print(f'Deleted ECR: {ECR_REPO}')
except Exception as e:
    print(f'ECR cleanup: {e}')
